# Step 2: split the data and find the score to beat

Before training anything: decide which ECGs the model learns from, which ones we use to tune it, and which
ones we keep locked away for the final test. Then work out how well you'd do with no model at all.

In [1]:
import numpy as np
import pandas as pd
import ecg_common as ec

db = ec.load_labels()

## The split

PTB-XL comes with a `strat_fold` number from 1 to 10 for every ECG. Folds 1-8 train the model, fold 9 tunes it
(picking which version to keep, where to put the cut-off), and fold 10 is the final test, used once at the very
end. Folds 9 and 10 had their labels double-checked by a person, so the scores we report are on the most
reliable labels.

The folds keep each patient's ECGs together. If one patient's ECGs were in both training and testing, the model
could score well by recognizing the patient instead of the disease. Let's check that's true.

In [2]:
train, val, test = ec.split_masks(db)
for name, mask in [("train (folds 1-8)", train), ("tune (fold 9)", val), ("test (fold 10)", test)]:
    print(f"{name:18s} {mask.sum():6d} ECGs")

patients = {name: set(db.patient_id[mask]) for name, mask in [("train", train), ("tune", val), ("test", test)]}
print("patients in both train and tune:", len(patients["train"] & patients["tune"]))
print("patients in both train and test:", len(patients["train"] & patients["test"]))
print("patients in both tune and test: ", len(patients["tune"] & patients["test"]))

train (folds 1-8)   17418 ECGs
tune (fold 9)        2183 ECGs
test (fold 10)       2198 ECGs
patients in both train and tune: 0
patients in both train and test: 0
patients in both tune and test:  0


## The first task: MI vs. normal

Start simple: ECGs that show an MI (label 1) against ECGs that are normal and nothing else (label 0). ECGs with
other problems but no MI are left out for now. That makes the task easier than real life, where the model would
also see bundle branch blocks, hypertrophy and so on. Step 4 handles all 5 categories at once.

In [3]:
keep, y = ec.mi_vs_normal(db)
rows = []
for name, mask in [("train", train), ("tune", val), ("test", test)]:
    m = mask & keep
    rows.append({"split": name, "ECGs": m.sum(), "MI": int(y[m].sum()), "normal": int((1 - y[m]).sum()),
                 "% MI": round(100 * y[m].mean(), 1)})
pd.DataFrame(rows).set_index("split")

,ECGs,MI,normal,% MI
split,,,,
train,11622,4379,7243,37.700001
tune,1454,540,914,37.099998
test,1462,550,912,37.599998


## The score to beat

**Accuracy** is the obvious score, but it's easy to fool. About 62% of these ECGs are normal, so a "model" that
always says normal is right 62% of the time while catching zero MIs.

So we'll mostly use **AUC**. Line up every MI and every normal ECG by the model's score: AUC is the chance a
random MI scores higher than a random normal. 0.5 is a coin flip, 1.0 is perfect. It doesn't depend on where you
put the cut-off between "MI" and "normal", which is step 5's job.

A second baseline: how far does age alone get you? Older patients are more likely to have had an MI, so ranking
ECGs by age should beat a coin flip without looking at the heart at all.

In [4]:
m = val & keep
print(f"always say normal: accuracy {100 * (1 - y[m].mean()):.1f}%, MIs caught 0 of {int(y[m].sum())}, AUC 0.500")
print(f"rank by age alone: AUC {ec.auc(y[m], db.age.values[m]):.3f}")

always say normal: accuracy 62.9%, MIs caught 0 of 540, AUC 0.500
rank by age alone: AUC 0.813


## What this step showed

- No patient is in more than one split, so the test score will be honest.
- Always saying "normal" is right 63% of the time on the tuning fold and catches no MIs.
- Age alone gets an AUC of 0.81. The MI patients in PTB-XL are much older than the normal ones, so a model could
  look decent just by learning what an old person's ECG looks like. Ours has to clearly beat 0.81 to show it's
  actually reading the heart. Step 3 trains the first one.